In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from preprocessing.preprocessing_utils import nn_preprocess
from sklearn.model_selection import train_test_split

In [2]:
# Load Data
print("Loading data...")
train_full = pd.read_csv("../data/claims_train.csv")
test_raw = pd.read_csv("../data/claims_test.csv")

# Create Validation Split (Same as Reference)
train_subset, val_subset = train_test_split(train_full, test_size=0.2, random_state=42)

# Apply Preprocessing
X_train_df, y_train, w_train, scaler = nn_preprocess(train_subset, scaler=None)
train_cols = X_train_df.columns.tolist()

X_val_df, y_val, w_val, _ = nn_preprocess(val_subset, scaler=scaler, ref_columns=train_cols)
X_test_df, y_test, w_test, _ = nn_preprocess(test_raw, scaler=scaler, ref_columns=train_cols)

# Convert to Pure NumPy Arrays (Float32 for consistency)
X_train = X_train_df.values.astype(np.float32)
y_train = y_train.values.astype(np.float32).reshape(-1, 1)
w_train = w_train.values.astype(np.float32).reshape(-1, 1)

X_val = X_val_df.values.astype(np.float32)
y_val = y_val.values.astype(np.float32).reshape(-1, 1)
w_val = w_val.values.astype(np.float32).reshape(-1, 1)

X_test = X_test_df.values.astype(np.float32)
y_test = y_test.values.astype(np.float32).reshape(-1, 1)
w_test = w_test.values.astype(np.float32).reshape(-1, 1)

print(f"Data Loaded. Features: {X_train.shape[1]}")

Loading data...
Data Loaded. Features: 124


In [3]:
# ==========================================
# 1. MANUAL NEURAL NETWORK (FROM SCRATCH)
# ==========================================
class ManualNN:
    def __init__(self, input_dim, hidden_layers, dropout_rate=0.0):
        """
        Initializes weights using He Initialization.
        """
        self.layers = []
        self.params = {}
        self.grads = {}
        self.cache = {}
        self.dropout_rate = dropout_rate
        self.training = True
        
        # --- Initialize Weights ---
        # We store architecture as a list of dimensions: [Input, Hidden1, Hidden2, ..., Output]
        layer_dims = [input_dim] + hidden_layers + [1]
        
        for i in range(1, len(layer_dims)):
            # He Initialization (Good for ReLU)
            scale = np.sqrt(2.0 / layer_dims[i-1])
            self.params[f'W{i}'] = np.random.randn(layer_dims[i-1], layer_dims[i]) * scale
            self.params[f'b{i}'] = np.zeros((1, layer_dims[i]))
            
        self.num_layers = len(layer_dims) - 1

        # --- Adam Optimizer State ---
        self.m = {}
        self.v = {}
        for key in self.params:
            self.m[key] = np.zeros_like(self.params[key])
            self.v[key] = np.zeros_like(self.params[key])
        self.t = 0 # Time step

    def forward(self, X):
        """
        Forward pass: Linear -> ReLU -> Dropout -> ... -> Linear -> Exp
        """
        A = X
        self.cache['A0'] = X
        
        for i in range(1, self.num_layers):
            W = self.params[f'W{i}']
            b = self.params[f'b{i}']
            
            # Linear
            Z = np.dot(A, W) + b
            self.cache[f'Z{i}'] = Z
            
            # ReLU
            A = np.maximum(0, Z)
            
            # Dropout (Inverted Dropout)
            if self.training and self.dropout_rate > 0:
                mask = (np.random.rand(*A.shape) > self.dropout_rate) / (1.0 - self.dropout_rate)
                A = A * mask
                self.cache[f'mask{i}'] = mask
            
            self.cache[f'A{i}'] = A
            
        # Final Layer (Linear -> Exponential for Poisson)
        W_last = self.params[f'W{self.num_layers}']
        b_last = self.params[f'b{self.num_layers}']
        
        Z_last = np.dot(A, W_last) + b_last
        self.cache[f'Z{self.num_layers}'] = Z_last
        
        # Exponential Activation (Force positive rate)
        A_last = np.exp(Z_last)
        self.cache[f'A{self.num_layers}'] = A_last
        
        return A_last

    def backward(self, y_true, w_expo, reg_lambda=0.0):
        """
        Backward pass for Poisson Deviance Loss.
        """
        m = y_true.shape[0]
        L = self.num_layers
        A_last = self.cache[f'A{L}']
        
        # --- Gradient at Output ---
        # For Poisson Loss with Log-Link (Exp activation), the gradient simplifies beautifully:
        # dL/dZ = w * (y_pred - y_true)
        # We divide by m because we minimize the MEAN loss.
        dZ = w_expo * (A_last - y_true) / m
        
        self.grads[f'dW{L}'] = np.dot(self.cache[f'A{L-1}'].T, dZ) + (reg_lambda * self.params[f'W{L}'])
        self.grads[f'db{L}'] = np.sum(dZ, axis=0, keepdims=True)
        
        # --- Backpropagate through hidden layers ---
        for i in range(L-1, 0, -1):
            dA = np.dot(dZ, self.params[f'W{i+1}'].T)
            
            # Apply Dropout Mask backward
            if self.training and self.dropout_rate > 0:
                dA = dA * self.cache[f'mask{i}']
            
            # Derivative of ReLU: 1 if Z > 0 else 0
            Z = self.cache[f'Z{i}']
            dZ = dA * (Z > 0)
            
            # Gradients for W and b
            prev_A = self.cache[f'A{i-1}']
            self.grads[f'dW{i}'] = np.dot(prev_A.T, dZ) + (reg_lambda * self.params[f'W{i}'])
            self.grads[f'db{i}'] = np.sum(dZ, axis=0, keepdims=True)

    def step_adam(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        """
        Manual implementation of Adam Optimizer step.
        """
        self.t += 1
        
        for key in self.params:
            grad = self.grads[f'd{key}']
            
            # Update biased first moment estimate
            self.m[key] = beta1 * self.m[key] + (1 - beta1) * grad
            
            # Update biased second raw moment estimate
            self.v[key] = beta2 * self.v[key] + (1 - beta2) * (grad ** 2)
            
            # Compute bias-corrected first moment estimate
            m_hat = self.m[key] / (1 - beta1 ** self.t)
            
            # Compute bias-corrected second raw moment estimate
            v_hat = self.v[key] / (1 - beta2 ** self.t)
            
            # Update parameters
            self.params[key] -= lr * m_hat / (np.sqrt(v_hat) + epsilon)

    def train_mode(self):
        self.training = True
        
    def eval_mode(self):
        self.training = False

In [4]:
# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def poisson_deviance_numpy(y_pred, y_true, weight):
    """
    Calculates Mean Poisson Deviance (Evaluation Metric)
    """
    y_pred = np.clip(y_pred, 1e-10, None) # Avoid log(0)
    term1 = np.zeros_like(y_true)
    mask = y_true > 0
    term1[mask] = y_true[mask] * np.log(y_true[mask] / y_pred[mask])
    term2 = y_true - y_pred
    deviance = 2 * weight * (term1 - term2)
    return np.mean(deviance)

In [5]:
# ==========================================
# 3. TRAINING LOOP
# ==========================================
def train_manual_model(config, lr, reg, epochs=500, batch_size=1024):
    input_dim = X_train.shape[1]
    
    # Initialize Model
    model = ManualNN(input_dim, config, dropout_rate=0.2)
    
    n_samples = X_train.shape[0]
    
    best_val_loss = float('inf')
    best_params = None
    patience = 15
    counter = 0
    history = []
    
    print(f"Training Manual NN {config} | LR: {lr} | Reg: {reg}")
    
    for epoch in range(epochs):
        # Shuffle
        indices = np.random.permutation(n_samples)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]
        w_shuffled = w_train[indices]
        
        model.train_mode()
        
        # Mini-batch Loop
        for i in range(0, n_samples, batch_size):
            end = min(i + batch_size, n_samples)
            X_batch = X_shuffled[i:end]
            y_batch = y_shuffled[i:end]
            w_batch = w_shuffled[i:end]
            
            # Forward / Backward / Step
            y_pred_batch = model.forward(X_batch)
            model.backward(y_batch, w_batch, reg_lambda=reg)
            model.step_adam(lr=lr)

        # --- Validation ---
        model.eval_mode()
        val_pred = model.forward(X_val)
        val_loss = poisson_deviance_numpy(val_pred, y_val, w_val)
        history.append(val_loss)
        
        # --- Early Stopping Logic ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            # Save best params (Deep copy is crucial here!)
            best_params = copy.deepcopy(model.params)
        else:
            counter += 1
            if counter >= patience:
                print(f"   Early stopping at epoch {epoch+1}")
                break
                
        if (epoch+1) % 10 == 0:
            print(f"   Epoch {epoch+1}: Val Loss {val_loss:.5f}")
            
    # Restore best weights
    if best_params is not None:
        model.params = best_params
        
    return model, best_val_loss, history

In [6]:
# ==========================================
# 4. EXECUTION & EVALUATION
# ==========================================

# 1. Define Best Architecture (from your Reference findings, e.g., [64, 32])
# If you haven't run Reference yet, start with something sensible:
winner_arch = [64, 32] 
learning_rate = 0.001
regularization = 0.001

# 2. Train
manual_model, final_val_loss, loss_history = train_manual_model(
    winner_arch, learning_rate, regularization, epochs=100
)

# 3. Final Test Prediction
manual_model.eval_mode()
y_pred_test = manual_model.forward(X_test)

# 4. Metrics
test_deviance = poisson_deviance_numpy(y_pred_test, y_test, w_test)

# Calculate D2 Score
global_mean = np.average(y_test, weights=w_test.flatten())
y_null = np.full_like(y_test, global_mean)
null_deviance = poisson_deviance_numpy(y_null, y_test, w_test)
d2_score = 1 - (test_deviance / null_deviance)

print("\n" + "="*40)
print("MANUAL IMPLEMENTATION RESULTS")
print("="*40)
print(f"Architecture:   {winner_arch}")
print(f"Test Deviance:  {test_deviance:.5f}")
print(f"D^2 Score:      {d2_score:.2%}")
print("="*40)

# 5. Plot Loss
plt.plot(loss_history)
plt.title("Manual Model Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Poisson Deviance")
plt.show()

# 6. Basic Calibration Check
print("\nCalibration Check (Manual):")
obs_mean = np.average(y_test, weights=w_test.flatten())
pred_mean = np.average(y_pred_test, weights=w_test.flatten())
print(f"Observed Freq:  {obs_mean:.4f}")
print(f"Predicted Freq: {pred_mean:.4f}")
print(f"Ratio:          {pred_mean/obs_mean:.2f}")

Training Manual NN [64, 32] | LR: 0.001 | Reg: 0.001
   Epoch 10: Val Loss 0.31896
   Epoch 20: Val Loss 0.31791
   Epoch 30: Val Loss 0.31787
   Epoch 40: Val Loss 0.31818
   Early stopping at epoch 43


TypeError: Axis must be specified when shapes of a and weights differ.